In [ ]:
"""
Process raw CT scan into binary volume
"""

import numpy as np
import sys
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
sys.path.append("../../src")
import process_ct as pct

In [ ]:
# make volume

sample = 'sample_name'
dir = 'path/to/sample/data'
vol = pct.process_ct_data(dir)
np.save("../../data/detrital_zircon_ct/raw/"+str(sample)+".npy", vol)
print(vol.shape)
print(vol.dtype)

In [ ]:
# threshold

sample = 'FWZ-1-1_1p5um'
vol = np.load("../../processed_data/raw/"+str(sample)+".npy")
plt.hist(vol.flatten(), bins=100, range=(10000,60000))

In [ ]:
threshold = 20000
bin = (vol > threshold)
fig, ax = plt.subplots(1,2, figsize=(10,5))
ax[0].imshow(vol.max(axis=0), 'gray')
ax[1].imshow(bin.max(axis=0), 'gray')
plt.tight_layout()

In [ ]:
# FOR 1.5 micron center VOI scans
from skimage.measure import label, regionprops

flat_vol = vol.max(axis=0)

# find pixels outside circle
outside_circle = (flat_vol == flat_vol[0,0]).astype(np.uint8)
outside_circle_3D = np.broadcast_to(outside_circle[None, :, :], bin.shape)

# label binary volume
labeled = label(bin + outside_circle)
regions = regionprops(labeled)

mask = (labeled != regions[0].label)

# apply mask
masked_bin = mask * bin

# now remove standards (if needed) (for 1.5 micron data)
# coords = np.argwhere(bin.max(axis=0))
# x_coords, y_coords = coords.T

# slope = 2
# intercept = -1200
# rm_coords_1 = coords[y_coords < (x_coords * slope + intercept)]

# masked_bin = np.copy(masked_bin1)
# masked_bin[:,rm_coords_1[:,0], rm_coords_1[:,1]] = 0

fig, ax = plt.subplots(2,2, figsize=(10,10))
ax = ax.flatten()
ax[0].imshow(outside_circle, 'gray')
ax[1].imshow(mask.min(axis=0), 'gray')
# ax[2].imshow(masked_bin1.max(axis=0), 'gray')
# xs = np.linspace(-intercept / slope, ((masked_bin1.shape[2] -intercept) / slope) -1, 2)
# ax[2].plot(slope*xs + intercept, xs, color="red")
ax[3].imshow(masked_bin.max(axis=0), 'gray')
plt.tight_layout()

In [ ]:
np.save(f"../../processed_data/binary/{sample}_binary_{threshold}.npy", masked_bin)

In [ ]:
# FOR 3 micron full scans (continue to the end)

# remove standards
flat_bin = bin.max(axis=0)
plt.imshow(flat_bin, 'gray')

# # boundaries for removing standards
# ymin = 0
# ymax = vol.shape[1]
# xmin = 500 
# xmax = 1500 # vol.shape[2]

# plt.axhline(y=xmin,c='r',lw=0.5)
# plt.axhline(y=xmax,c='r',lw=0.5)

# xs = np.linspace(0, 1000, 10)
# s1 = 3
# i1 = 100
# plt.plot(s1*xs + i1, xs, color="red")

# xs = np.linspace(1000, 2000, 10)
# s2 = 3
# i2 = -3600
# plt.plot(s2*xs + i2, xs, color="red")

# circle
# Circle parameters
# xc1 = 1200       # x-coordinate of center
# yc1 = 1400       # y-coordinate of center
# r1 = 600             # radius

# # Generate circle points
# theta = np.linspace(0, 2*np.pi, 400)
# x = xc1 + r1 * np.cos(theta)
# y = yc1 + r1 * np.sin(theta)

# plt.plot(x,y,c='r')

# # Circle parameters
# xc0 = 750       # x-coordinate of center
# yc0 = 2150       # y-coordinate of center
# r0 = 200             # radius

# # Generate circle points
# theta = np.linspace(0, 2*np.pi, 400)
# x = xc0 + r0 * np.cos(theta)
# y = yc0 + r0 * np.sin(theta)

# plt.plot(x,y,c='r')

In [ ]:
from skimage.draw import disk

masked = bin
# # square
# mask = np.zeros_like(bin)
# mask[:,xmin:xmax,ymin:ymax] = 1

# # apply mask
# masked = bin*mask

# angular
# Remove anything to the left of the first line, and anything to the right of the second line

# get coordinates of all grain pixels
# coords = np.argwhere(flat_bin)
# x_coords, y_coords = coords.T

# rm_coords_1 = coords[y_coords > (x_coords * s1 + i1)]
# rm_coords_2 = coords[y_coords < (x_coords * s2 + i2)]

# masked = bin.copy()
# masked[:,rm_coords_1[:,0], rm_coords_1[:,1]] = 0
# masked[:,rm_coords_2[:,0], rm_coords_2[:,1]] = 0

# circles

# Grid dimensions
nz, nx, ny = np.shape(bin)   # size of array

# Create empty 2D mask
mask2d = np.zeros((nx, ny), dtype=bool)

# `disk` returns the pixel coordinates inside the circle
rr, cc = disk((yc1, xc1), r1, shape=(nx, ny))
mask2d[rr, cc] = True

# Broadcast the 2D mask along z to form a cylinder
mask3d_1 = np.repeat(mask2d[None, :, :], nz, axis=0)

# # second circle
# # Compute squared distance from center in x-y plane
# # Create empty 2D mask
# mask2d = np.ones((nx, ny), dtype=bool)

# # `disk` returns the pixel coordinates inside the circle
# rr, cc = disk((yc0, xc0), r0, shape=(nx, ny))
# mask2d[rr, cc] = False

# # Broadcast the 2D mask along z to form a cylinder
# mask3d_0 = np.repeat(mask2d[None, :, :], nz, axis=0)

# mask = mask3d_1*mask3d_0

# apply mask
masked = bin*mask3d_1

cropped, _ = pct._reduce_volume(masked, 10)
plt.imshow(cropped.max(axis=0), 'gray')

In [ ]:
np.save(f"../../processed_data/binary/{sample}_binary_{threshold}.npy", cropped)